# 4. Capa Silver

In [0]:
# Definición de catálogo y esquemas
catalog = "smart_Claims"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

In [0]:
# leemos las tablas bronze
main_Claims_bz= spark.table(f"{catalog}.{BRONZE_SCHEMA}.claims")

### 4.1 Transformación para main_Claims

In [0]:
display(main_Claims_bz)

### 4.2 Transformación para Customer

In [0]:
# Load customers table from bronze
customers = spark.table(f"{catalog}.{BRONZE_SCHEMA}.customers")
display(customers.limit(10))

In [0]:
df_customers =(
    spark.table(f"{catalog}.{BRONZE_SCHEMA}.customers")
    #Eliminar registros sin clave de negocio
    .filter(col("customer_id").isNotNull())
    #Eliminar duplciados
    .dropDuplicates(["customer_id"])
    #Convetir fecha de nacimiento para múltiples formatos
    .withColumn (
        F.coalesce(
            F.to_date(col("date_of_birth"), "yyyy-MM-dd"),
            F.to_date(col("date_of_birth"), "MM-dd-yyyy"),
            F.to_date(col("date_of_birth"), "dd-MM-yyyy"),
            
        )
    )
)
#Limpiar y estandarizar nombre
.withColumm(
    "customer_name", 
    F.initcap(F.trim(F.col("customer_name")))
)
#Seperar primer nombre
.withColumn(
    "first_name", 
    F.trim(F.split(F.col("customer_name"), r"\s+")[0])
)
#Separar apellido
.withColumn(
    "last_name", 
    F.trim(F.element_at(F.split(F.col("customer_name"), r"\s+"), -1))
)
#Construir dirección
.withColumn(
    "address"
    F.concat_ws(",",F.trim(F.col("borough")),F.trim(F.col("zip_code")))
)
.withColumn("_silver_ts",F.current_timestamp())
)
safe_write(df_customers,SILVER_SCHEMA,"customer_clean")
# Transformaciones de polizas
df_policies = (
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.policies")
    .filter(F.col("policy_id").isNotNull())
    .dropDuplicates(["policy_id"])
    .withColumn("premium", F.col("premium").cast("double"))
    .withColumn("_silver_ts", F.current_timestamp())
)
 
safe_write(df_policies, SILVER_SCHEMA, "policies_clean")
# Transformaciones según la guía
df_claims = (
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.claims")
    .filter(F.col("claim_no").isNotNull())
    .dropDuplicates(["claim_no"])
    .withColumn(
        "claim_date",
        F.coalesce(
            F.to_date("claim_date", "yyyy-MM-dd"),
            F.to_date("claim_date", "MM/dd/yyyy"),
        )
    )
    # incident_date → timestamp con nombre semántico incident_ts
    .withColumn(
        "incident_ts",
        F.coalesce(
            F.to_timestamp("incident_date", "yyyy-MM-dd HH:mm:ss"),
            F.to_timestamp("incident_date", "yyyy-MM-dd"),
        )
    )
)
#Eliminar la columna para evitar ambiguedad
 .drop("incident_date")
    .withColumn(
        "license_issue_date",
        F.coalesce(
            F.to_date("license_issue_date", "yyyy-MM-dd"),
            F.to_date("license_issue_date", "MM/dd/yyyy"),
        )
    )
    .withColumn("_silver_ts", F.current_timestamp())

 
safe_write(df_claims, SILVER_SCHEMA, "claims_clean")

# Transformaciones de telematics
df_telematics = (
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.telematics")
    # Parsear timestamp con formatos alternativos
    .withColumn(
        "event_ts",
        F.coalesce(
            F.to_timestamp("timestamp", "yyyy-MM-dd HH:mm:ss"),
            F.to_timestamp("timestamp", "yyyy-MM-ddTHH:mm:ss"),
            F.to_timestamp("timestamp"),
        )
    )
  
#Eliminar columna cruda para evitar duplicado
 .drop("timestamp")
 #Filtrar coodenadas inválidas o nulas
  .filter(
        F.col("latitude").isNotNull()
        & F.col("longitude").isNotNull()
        & F.col("latitude").between(-90, 90)
        & F.col("longitude").between(-180, 180)
    )
    # Campo derivado para particionamiento
    .withColumn("event_date", F.to_date("event_ts"))
    .withColumn("_silver_ts", F.current_timestamp())
)
#Particionamiento por fecha
safe_write(df_telematics, SILVER_SCHEMA, "telematics_clean", partition_by=["event_date"])
# Entrenamiento para Images
df_training = (
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.training_images")
    .withColumn(
        "image_name",
        F.element_at(F.split(F.col("path"), "/"), -1)
    )
    .withColumn(
        "label",
        F.split(F.col("image_name"), "_")[0]
    )
    # Solo conservar imágenes con label válido
    .filter(
        F.col("label").isNotNull()
        & (F.length(F.col("label")) > 0)
    )
    .withColumn("_silver_ts", F.current_timestamp())
)
 
safe_write(df_training, SILVER_SCHEMA, "training_images")
# Transformaciones Claim_metadata
df_claim_images = (
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.claim_images")
    .withColumn(
        "image_name",
        F.element_at(F.split(F.col("path"), "/"), -1)
    )
    # Normalizar nombre para joins robustos (lowercase, sin espacios)
    .withColumn(
        "image_name",
        F.lower(F.trim(F.col("image_name")))
    )
    .withColumn("_silver_ts", F.current_timestamp())
)
 
safe_write(df_claim_images, SILVER_SCHEMA, "claim_images")
# Tranformaciones para claim metadata
df_claim_metadata = (
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.claim_metadata")
    .filter(
        F.col("calim_no").isNotNull()
        & F.col("image_name").isNotNull()
    )
.withColumn(
        "image_name",
        F.lower(F.trim(F.col("image_name")))
    )
    .dropDuplicates(["image_name"])
    .withColumn("_silver_ts", F.current_timestamp())
)
safe_write(df_metadata, SILVER_SCHEMA, "claim_images_metadata_clean")

# Validación Silver
print("=" * 65)
print("VALIDACIÓN — CAPA SILVER")
print("=" * 65)
 
silver_checks = {
    "customers_clean": {
        "required_cols": ["customer_id", "firstname", "lastname",
                          "address", "birth_date"],
        "date_cols":     ["birth_date"],
        "key":           "customer_id",
    },
    "policies_clean": {
        "required_cols": ["policy_id", "customer_id", "chassis_no", "premium"],
        "date_cols":     [],
        "key":           "policy_id",
    },
    "claims_clean": {
        "required_cols": ["claim_no", "policy_id", "claim_date",
                          "incident_ts", "license_issue_date"],
        "date_cols":     ["claim_date", "incident_ts", "license_issue_date"],
        "key":           "claim_no",
    },
    "telematics_clean": {
        "required_cols": ["chassis_no", "event_ts", "event_date",
                          "latitude", "longitude"],
        "date_cols":     ["event_ts"],
        "key":           "chassis_no",
    },
    "training_images": {
        "required_cols": ["image_name", "label", "content"],
        "date_cols":     [],
        "key":           "image_name",
    },
    "claim_images": {
        "required_cols": ["image_name", "content"],
        "date_cols":     [],
        "key":           "image_name",
    },
    "claim_metadata_clean": {
        "required_cols": ["image_name", "claim_no", "chassis_no"],
        "date_cols":     [],
        "key":           "image_name",
    },
}
all_ok = True
for table, cfg in silver_checks.items():
    fqn = f"{CATALOG}.{SILVER_SCHEMA}.{table}"
    print(f"\n  {fqn}")
    try:
        df    = spark.table(fqn)
        total = df.count()
        print(f"     Filas       : {total:,}")
 
        # Nulos en columna clave
        nulls = df.filter(F.col(cfg["key"]).isNull()).count()
        print(f"     Nulos clave : {nulls} {'' if nulls == 0 else ''}")
 
        # Fechas nulas
        for dc in cfg.get("date_cols", []):
            if dc in df.columns:
                n   = df.filter(F.col(dc).isNull()).count()
                pct = round(n / total * 100, 1) if total else 0
                print(f"     Nulos {dc:<24}: {n} ({pct}%) {'' if n == 0 else ''}")
# Columnas requeridas
 actual  = set(df.columns)
        missing = set(cfg["required_cols"]) - actual
        if missing:
            print(f"Columnas faltantes: {missing}")
            all_ok = False
        else:
            print(f"Todas las columnas requeridas presentes")
 
    except Exception as e:
        print(f"     Error: {e}")
        all_ok = False
 
print("\n" + "=" * 65)
print(f"Resultado: {'Silver OK' if all_ok else 'Revisa los campos marcados'}")